# 06 · Line profiles and optical depth

This notebook goes from the optical-depth integral along a path to the sum that SkewerJAX evaluates, and checks every
ingredient of that sum. In order:

1. the derivation from the integral to the sum over pieces, with a convergence test;
2. the Voigt function;
3. the profiles at points;
4. closed-form checks;
5. the line-centred frames, the global velocity axis and the observed wavelength;
6. periodicity, an option of the optical depth;
7. an application to simulation outputs.

Runtime: ~1 min.

**Requires:** lines (01), skewers (03, $a(x)$ from 02) and fields (04); the last section also simulation outputs (05).  
**Independent of:** the summary statistics (07).

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import numpy as np, jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)          # SkewerJAX follows this flag; it never sets it itself
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import use_style, use_backend
# use_backend()      # interactive figures (ipympl) in a live notebook; inline when executed headless or without ipympl
use_style()          # paper style (white); use_style("dark") for the dark one: same colour-blind-safe palette in both
from skewerjax.utils.plotting import (ENTITY_COLORS as EC, MODE_STYLES as MS, PALETTE as TOL, FG, BG, DIM, CMAPS, LABELS,
                                      FIGSIZES, add_panel_letters, plot_dag, plot_spectra, zoom_inset)

In [ ]:
plot_dag(('lines', 'cosmology', 'skewers', 'samplers', 'simulation', 'fields', 'tau',)); plt.show()

In [ ]:
from skewerjax.core import (LineBundle, LINES, voigt as V, line_wavelength_grid_A, frequency_offset_GHz, line_profiles,
                            compute_optical_depth, resolution_report, SkewerBundle, Fields,
                            ScalarField, VectorField, LogRamp, VelocityRamp, Constant, plane_points, analytic as an, phys)
lines = LineBundle(["HI-LyA", "HI-LyB", "SiIII-1206"], ref_line_id="HI-LyA")
print(f"{len(LINES)} transitions in lines_table.dat")
for l, dv in zip(lines.lines, np.asarray(lines.delta_v)):
    print(f"  {l.id:11s} lambda0 = {l.lambda0_A:9.4f} A  f = {l.f_osc:.4f}  A21 = {l.A21_Hz:.3e} Hz  m = {l.mass_amu:7.3f} amu  delta_v = {dv:9.1f} km/s")
from skewerjax.core import Cosmology

## 1 · From the integral to the sum over pieces

**The integral.** Along a straight comoving path, a photon is absorbed by transition $j$ of species $i$ with optical
depth (review notes, Eq. `tau_straight_los`, with $dl_{\rm phys} = a\, dx$)

$$
\tau_j(v_\dagger) = \mathcal{C}_j \int \frac{n_{\rm com}(x)}{a(x)^3}\, \phi_j\big(\nu_j(v_\dagger, x)\big)\, a(x)\, dx
= \mathcal{C}_j \int \frac{n_{\rm com}(x)}{a(x)^2}\, \phi_j\big(\nu_j(v_\dagger, x)\big)\, dx,
\qquad \mathcal{C}_j = \frac{e^2 f_j}{4\varepsilon_0 m_e c} .
$$

It is a line integral along the path of the light.

**The spectral coordinate.** The observer labels every photon by its line-centred velocity $v_\dagger$. At $x$, a gas
element with scale factor $a(x)$ and velocity $v_\parallel$ along the photon sees it at the local wavelength

$$
\lambda'_j = \lambda_j\, e^{v_\dagger/c}\, \frac{a(x)}{a_\dagger}\, D^{-1}(x),
\qquad D^{-1} = \sqrt{\frac{1+\beta}{1-\beta}},\qquad \beta = \frac{v_\parallel}{c}
$$

(review notes, Eq. `velocity_units_normalized`). The element absorbs at line centre ($\lambda'_j = \lambda_j$) when
$v_\dagger$ equals

$$
v_c(x) = c\left[\ln\frac{a_\dagger}{a(x)} + \frac12 \ln\frac{1-\beta}{1+\beta}\right] .
$$

Its frequency detuning is then an exact function of $v_\dagger - v_c$:

$$
\nu_j = c\left(\frac{1}{\lambda'_j} - \frac{1}{\lambda_j}\right) = \frac{c}{\lambda_j}\left[e^{-(v_\dagger - v_c)/c} - 1\right] .
$$

**Step 1: cut the path into pieces.** The integral is the sum of the integrals over the parts of any partition of the
path. SkewerJAX uses **pieces** $p$, of length $\ell_p$: fragments of the path within which the gas properties are
constant. Every field source reaches the optical depth the same way, as pieces (`core.Pieces`); the sources differ
only in how they cut the path.

- **Simulation outputs** (05). The gas is one value per Voronoi cell, so the pieces are the exact intersections of the
  path with the cells. By default they are also cut at the ends of the dual segments of the sampling points, one piece
  per cell and dual segment; the cell crossings (`deposition="voronoi_crossings"`) keep one interval per crossing of a
  cell, split into equal sub-pieces.
- **Fields known everywhere** (the samplers of 04). There are no cells. Each point $x_i$ stands for its *dual segment*
  $[m_{i-1/2}, m_{i+1/2}]$, of length $w_i$ (03), and the dual segment is the piece; `subdivide=q` cuts it into $q$ equal
  pieces of length $w_i/q$ (`core.pieces_from_fields`). A field given only at the points (`SampledField`) gives every
  piece the value of its point.

**Step 2: what is constant within a piece, and what is not.** Constant: $n_{\rm com}$, $T$ (and so $\sigma$ and
$\gamma$) and the velocity vector; the path is straight within a piece, so $v_\parallel = \mathbf{v}\cdot\hat t$ is
constant too. For simulation pieces this is exact (the gas of one cell); for fields known everywhere it is the
approximation that the fields vary little across a piece. Not constant: $a(x)$, so $a^{-2}$ and the Hubble part of the
line centre, $v_H(x) = c\ln(a_\dagger/a(x))$, change across every piece. The line centre moves across a piece by

$$
W_p = \left\lvert\frac{dv_c}{dx}\right\rvert \ell_p,
$$

where $dv_c/dx$ is the Hubble rate $dv_H/dx$, plus the gradient of $v_\parallel$ for a field known everywhere.

**Step 3: one profile per piece, at its centroid.** The scale factor $a$, $a^{-2}$, $v_c$ and the fields are all
evaluated at the centroid $s_p$ of every piece, for every source (for fields known everywhere, $a$ is interpolated in
$\ln a$ between the bundle's points). This is the midpoint rule on each piece:

$$
\tau_j(v_\dagger) \simeq \mathcal{C}_j \sum_p \frac{n_{{\rm com},p}\, \ell_p}{a_p^2}\, \phi_{j,p}(v_\dagger),
\qquad \phi_{j,p}(v_\dagger) = V\big(\nu_j(v_\dagger - v_{c,p});\, \sigma_p, \gamma_p\big),
$$

where:

- $\sigma_p = b_p/(\sqrt2\, \lambda_j)$, with $b_p = \sqrt{2k_BT_p/m} + b_{\rm turb}$;
- $\gamma_p = (A_{21} + A_{\rm coll})/(4\pi)$;
- $V$ is the Voigt function (§2).

This is `core.optical_depth_kernel`, mapped over skewers by `optical_depth_from_pieces`, behind
`compute_optical_depth(source, v, lines, ...)`, whose source is a `SkewerBundle` with its `Fields`, or `Pieces`.

**Error, and when the sum is exact.**

- The midpoint rule replaces the profile smeared over $W_p$ by the profile at its centre. For a Gaussian core this
  changes the peak by a relative $W_p^2/(12 b^2)$: the sum converges to the integral at second order in the piece
  length.
- $\phi$ has unit area in $\nu$, so $\int \tau\, d\nu = \mathcal{C}_j \sum_p n_{{\rm com},p}\, \ell_p\, a_p^{-2}$ for any
  partition. For simulation pieces the column $\sum_p n_p \ell_p$ is exact, because $n$ is exactly constant in a cell.
  For fields known everywhere it is the midpoint rule of $\int n\, dx$, exact only where $n$ is constant across the
  piece.
- The sum equals the integral exactly when $a$, $v_c$ (and, for fields known everywhere, the fields) are constant
  within every piece. It converges to it as $\max_p W_p/b_p \to 0$.
- The resolution check of `compute_optical_depth` (`resolution_report`) monitors exactly this ratio, with the width of
  every piece (`Pieces.width_kms`). With the simulation default of 0.25 km/s spacing, $W_p \le 0.25$ km/s, far below
  any thermal width.

**A test.** A toy density along an unevenly sampled skewer (61 points, $z = 3$, a peculiar-velocity gradient), with
1 to 8 pieces per dual segment against 64.

In [ ]:
from skewerjax.core import pieces_from_fields, PowerLawGRF, GRFDensity
cos_p = Cosmology(H0=phys.H0, Omega_r_0=0.0, Omega_m_0=0.31, Omega_k_0=0.0, Omega_lambda_0=0.69)
Lp = 10.0
s_u = Lp * np.linspace(0.0, 1.0, 61) ** 1.5                                   # 61 unevenly spaced points along x [cMpc]
sb_p = SkewerBundle(spatial_points=jnp.asarray(np.stack([s_u, np.full_like(s_u, 3.0), np.full_like(s_u, 4.0)], -1)[None]),
                    a0=0.25, x0=Lp, cosmologies=cos_p)                          # observer beyond the last point, z = 3
grf_p = PowerLawGRF(A_s=0.01, k_0=2.0, n_s=-1.25, L=(Lp,) * 3, N=(32,) * 3)
f_p = Fields({"HI": {"temperature": ScalarField(Constant(1.5e4)), "velocity": VectorField(VelocityRamp(-60.0, 60.0, Lp)),
                     "number_density": ScalarField(GRFDensity(jax.random.PRNGKey(5), grf_p, n_bar=2e-12)),
                     "turbulence": ScalarField(Constant(0.0)), "collisional": ScalarField(Constant(0.0))}})
lya1 = LineBundle(["HI-LyA"], ref_line_id="HI-LyA")
v_p = np.linspace(-150.0, 1000.0, 2301)
tq = {q: np.asarray(compute_optical_depth(sb_p, v_p, lya1, f_p, a_dagger=0.25, subdivide=q, resolution_tolerance=None))[0, 0]
      for q in (1, 2, 4, 8, 64)}
p1 = pieces_from_fields(sb_p, f_p, lya1, a_dagger=0.25)                       # one piece per dual segment
rep = resolution_report(p1, lya1)
err = {q: np.abs(tq[q] - tq[64]).max() / tq[64].max() for q in (1, 2, 4, 8)}
fig, axs = plt.subplots(1, 2, figsize=(22, 6.5), layout="constrained", gridspec_kw={"width_ratios": [2.2, 1]})
axs[0].plot(v_p, tq[64], color=FG, lw=3, alpha=0.35, label="64 pieces per dual segment (reference)")
for q, c in ((1, TOL["blue"]), (4, TOL["red"])):
    axs[0].plot(v_p, tq[q], color=c, lw=1.6, label=f"{q} piece{'s' if q > 1 else ''} per dual segment")
axs[0].set(xlabel=LABELS["v_dagger"], ylabel=r"$\tau$ ($\mathrm{Ly}\alpha$)"); axs[0].legend()
qs = np.array([1, 2, 4, 8])
axs[1].loglog(qs, [err[q] for q in qs], "o-", color=TOL["blue"], label="measured")
axs[1].loglog(qs, err[1] * qs ** -2.0, ls="--", color=FG, label=r"$\propto q^{-2}$")
axs[1].set(xlabel=r"pieces per dual segment $q$", ylabel=r"max $|\tau_q - \tau_{64}|/\tau_{\max}$"); axs[1].legend()
add_panel_letters(axs); fig.suptitle("Pieces of a field known everywhere: the sum converges at second order in the piece length", fontsize=21)
plt.show()
print(f"one piece per dual segment: max W/b = {rep['max_ratio'][0]:.2f}, column fraction above b/4 = {rep['column_fraction'][0]:.2f}")
print("max |tau_q - tau_64| / tau_max:", {q: f"{e:.1e}" for q, e in err.items()},
      "| integral of tau dv relative to q = 64:", {q: f"{tq[q].sum() / tq[64].sum() - 1:+.1e}" for q in (1, 2, 4, 8)})

*Reading the figure.* With one piece per dual segment the widest pieces span 1.4 thermal widths ($W/b = 1.44$),
and the profile is visibly smeared. The error falls towards $q^{-2}$ as the pieces shrink, as the midpoint rule
predicts, and the integral of $\tau$ changes far less than the profile, because only the density's own variation
across a piece enters it.

## 2 · The Voigt function

Three implementations of the same function $V(x; \sigma, \gamma) = (G_\sigma * L_\gamma)(x)$, the convolution of a
Gaussian of standard deviation $\sigma$ and a Lorentzian of half-width $\gamma$:

- `"ExoJAX"` (default): a Humlíček-type approximation of the Faddeeva function, compatible with `jit` and `grad`;
- `"convolution"`: $G * L$ by FFT on the grid, refined 4 times; it needs a grid that resolves both widths;
- `"scipy_wofz"`: the reference, in NumPy, with

$$
V = \frac{\mathrm{Re}\, w(z)}{\sqrt{2\pi}\, \sigma},\qquad z = \frac{x + i\gamma}{\sqrt2\, \sigma} .
$$

The four $(\sigma, \gamma)$ pairs combine $10^{-3}$ and $1$:

- the Gaussian limit ($\gamma \ll \sigma$);
- the Lorentzian limit ($\sigma \ll \gamma$);
- two Voigt profiles with both widths equal.

In [ ]:
from skewerjax.utils.plotting import plot_voigt_limits
x = jnp.linspace(-9, 9, 2 ** 13)
sigma = np.array([1e-3, 1.0, 1e-3, 1.0]); gamma = np.array([1e-3, 1e-3, 1.0, 1.0])
curves = {"scipy_wofz": np.stack([V.voigt_scipy(np.asarray(x), s, g) for s, g in zip(sigma, gamma)]),
          "ExoJAX": np.stack([np.asarray(V.voigt_exojax(x, s, g)) for s, g in zip(sigma, gamma)]),
          "convolution (FFT, oversample 4)": np.stack([np.asarray(V.voigt_convolution(x, s, g, oversample=4)) for s, g in zip(sigma, gamma)])}
fig, axs = plot_voigt_limits(np.asarray(x), sigma, gamma, curves)
add_panel_letters(axs); fig.suptitle("Voigt: the Gaussian and Lorentzian limits; ExoJAX (solid) and FFT convolution (dashed) on scipy's wofz", fontsize=21)
plt.show()
for name in ("ExoJAX", "convolution (FFT, oversample 4)"):
    r = np.abs(curves[name] / curves["scipy_wofz"] - 1)
    core = np.abs(np.asarray(x))[None] < 3 * np.maximum(sigma, gamma)[:, None]
    print(f"{name:32s} max |V / wofz - 1| per pair (|x| < 3 max(sigma, gamma)):", np.array2string(np.where(core, r, 0).max(1), precision=1))

ExoJAX agrees with `wofz` to $\sim 10^{-6}$ or better for every pair. The convolution is exact ($10^{-14}$) when the
grid resolves both widths, and it fails where the grid cannot:

- $\sigma = \gamma = 10^{-3}$ is far below the grid spacing ($2.2\times10^{-3}$);
- the Lorentzian wings are cut at the grid ends.

Full checks: `scripts/validation/check_voigt.py`.

## 3 · Line profiles at points

`line_profiles(events, directions, v, a_†, lines, fields)` evaluates $\phi_{ij}$ for every line and every point.

- A hotter gas broadens the lines.
- A peculiar velocity along the photon shifts them: $v_c \approx c\ln(a_\dagger/a) - v_\parallel$, where $v_\parallel$
  is measured along the photon direction. Gas moving with the photon, i.e. towards the observer, is blueshifted to
  smaller $v_\dagger$.
- In panel (b) the photons travel along $+x$, so the points with $v_x > 0$ absorb at negative $v_\dagger$.

In [ ]:
pts = plane_points(1e-3, "y", 5e-4, 1, 5)[0]
events = jnp.concatenate([jnp.ones((5, 1)), pts], -1)
v = jnp.linspace(-400, 400, 801)
fig, axs = plt.subplots(1, 2, figsize=(20, 6), layout="constrained")
for ax, (Tlo, Thi, vlo, vhi, title) in zip(axs, ((1e3, 1e6, 0.0, 0.0, r"$T$ from $10^3$ to $10^6$ K"), (1e4, 1e4, -300.0, 300.0, r"$v_x$ from −300 to +300 km/s, $T = 10^4$ K"))):
    sp = {"temperature": ScalarField(LogRamp(Tlo, Thi, 1e-3, 3)), "velocity": VectorField(VelocityRamp(vlo, vhi, 1e-3)),
          "turbulence": ScalarField(Constant(0.0)), "collisional": ScalarField(Constant(0.0)), "number_density": ScalarField(Constant(1.0))}
    ev = events.at[:, 3].set(jnp.linspace(0, 1e-3, 5)) if Tlo != Thi else events.at[:, 1].set(jnp.linspace(0, 1e-3, 5))
    phi = np.asarray(line_profiles(ev, jnp.broadcast_to(jnp.array([1.0, 0, 0]), (5, 3)), v, 1.0, LineBundle(["HI-LyA"]), Fields({"HI": sp})))[0]
    for i in range(5):
        ax.semilogy(v, phi[i], color=list(TOL.values())[i])
    ax.set(xlabel=r"$v_\dagger$ [km/s]", ylabel=r"$\phi$ [ns]", ylim=(1e-8, None), title=title)
add_panel_letters(axs); fig.suptitle(r"Ly$\alpha$ profiles: temperature sets the width, the line-of-sight velocity the centre", fontsize=22); plt.show()

The detuning $\nu_j$ of §1 uses the exact exponential mapping by default; `detuning="linear"` replaces it by its first
order, an option kept for comparisons with other codes. `velocity_period` replaces $v_\dagger - v_c$ by its nearest
image modulo a period (§6).

## 4 · Optical depth: checks against closed forms

Static uniform gas ($a = 1$, no velocities) has closed forms (review notes):

- a Lorentzian for $T = 0$;
- a Gaussian for $A_{21} = 0$, the `null...` lines of the table.

Lines of one species share the fields.

In [ ]:
Lk = 1e-3
def fields(T, n):
    sp = lambda: {"temperature": ScalarField(LogRamp(*T, Lk, 3)), "velocity": VectorField(VelocityRamp(0.0, 0.0, Lk)),
                  "number_density": ScalarField(LogRamp(*n, Lk, 2)), "turbulence": ScalarField(Constant(0.0)), "collisional": ScalarField(Constant(0.0))}
    return Fields({"HI": sp(), "SiIII": sp()})
null = LineBundle(["nullHI-LyA", "nullHI-LyB", "nullSiIII1206"])
pts = plane_points(Lk, "y", Lk / 2, 4, 32)                                       # 4 skewers, temperature rising with z
sb = SkewerBundle(spatial_points=pts, manual_scale_factors=jnp.ones_like(pts[None, ..., 0]))
v = jnp.linspace(-3000.0, 3000.0, 1024)
F_th = fields((1e4, 1e8), (1e-7, 1e-7))
tau = compute_optical_depth(sb, v, null, F_th, a_dagger=1.0)
T_sk = F_th["HI"]["temperature"](sb.quadri_vectors[0])[:, 0]
wl = line_wavelength_grid_A(v, null, 1.0, 1.0)
ref = jnp.stack([an.tau_static_thermal(wl[j], null.lambda0_A[j:j+1], null.mass_amu[j:j+1], null.f_osc[j:j+1], sb.a0, Lk * jnp.ones(4),
                                       1e-7 * jnp.ones(4), T_sk)[0] for j in range(3)])
F, Fa = np.exp(-np.asarray(tau)), np.exp(-np.asarray(ref))
fig, axs = plt.subplots(2, 3, figsize=(22, 8), sharex=True, gridspec_kw={"height_ratios": [2, 1]}, layout="constrained")
for j, l in enumerate(lines.lines):
    for k in range(4):
        c = list(TOL.values())[k]
        axs[0, j].plot(v, Fa[j, k], color=c, **MS["reference"])
        axs[0, j].plot(v, F[j, k], color=c, **MS["default"], label=f"T = {float(T_sk[k]):.0e} K" if j == 0 else None)
        axs[1, j].plot(v, F[j, k] - Fa[j, k], color=c)
    axs[0, j].set_title(l.name); axs[1, j].set_xlabel(r"$v_\dagger$ [km/s]")
axs[0, 0].set_ylabel("F"); axs[1, 0].set_ylabel(r"$F - F_{\rm analytic}$"); axs[0, 0].legend(fontsize=15)
add_panel_letters(axs); fig.suptitle("Static thermal lines: SkewerJAX (solid) on the analytic Gaussians (light); residuals < 10⁻⁸", fontsize=22); plt.show()

Radiation-dominated expansion and the convergence with the sampling are checked in
`scripts/validation/check_optical_depth_analytic.py`.

## 5 · Line-centred frames, the global velocity axis and the observed wavelength

These are three axes for the same spectrum; changing from one to another is a change of axis only.

- **The line-centred frame** of every line: each line $j$ is integrated on a grid of $v_\dagger$, its own frame. By §1, the
  line centre of the gas at $x$ is the same $v_c(x)$ in every frame: it does not depend on $\lambda_j$
  (`core.spectral.line_centre_velocity_kms`).
- **The global velocity axis** of a reference line (here $\mathrm{Ly}\alpha$): the same observed wavelength sits at
  $v_{\rm Ly\alpha} = v_\dagger - \Delta v_j$ in the reference frame, with

$$
\Delta v_j = c\ln\frac{\lambda_{\rm Ly\alpha}}{\lambda_j}
$$

  (`LineBundle.delta_v`). `compute_optical_depth(..., line_frame="reference")` evaluates every line directly on that
  axis, line $j$ at $v + \Delta v_j$ in its own frame (`spectral.velocities_in_line_frames`).
- **The observed wavelength** (`spectral.line_wavelength_grid_A`):

$$
\lambda_{\rm obs} = \lambda_j\, \frac{a_{\rm obs}}{a_\dagger}\, e^{v_\dagger/c} .
$$

**The gas.** Four clumps along a 20 cMpc skewer at $z \approx 3$, with no peculiar velocities. Each clump has its own
width, peak density and temperature ($8\times10^3$ to $6\times10^4$ K), so that every absorber is easy to tell apart.
The lines are $\mathrm{Ly}\alpha$ and Si III $\lambda$1206, with Si III 5 times less abundant to make it visible.

In [ ]:
from skewerjax.core import Cosmology, SampledField
from skewerjax.utils.plotting import plot_frames_axes
eds = Cosmology(H0=phys.H0, Omega_r_0=0.0, Omega_m_0=1.0, Omega_k_0=0.0, Omega_lambda_0=0.0)
L20, n_pt, z_obs = 20.0, 8193, 3.0
xs = np.linspace(0.0, L20, n_pt)
pts = jnp.asarray(np.stack([xs, np.full_like(xs, 1.0), np.full_like(xs, 1.0)], -1)[None])
sbf = SkewerBundle(spatial_points=pts, cosmologies=eds, a0=1 / (1 + z_obs), x0=L20)      # observer beyond the far end
# four clumps (position [cMpc], width [cMpc], peak density [cm^-3], temperature [K]): each one different
clumps = [(3.0, 0.12, 1.2e-12, 8e3), (7.5, 0.25, 0.4e-12, 6e4), (12.0, 0.10, 2.0e-12, 1.5e4), (16.5, 0.20, 0.8e-12, 3e4)]
g = np.stack([n0 * np.exp(-0.5 * ((xs - c0) / w0) ** 2) for c0, w0, n0, _ in clumps])
n_x = g.sum(0) + 1e-14
T_x = (g * np.array([c[3] for c in clumps])[:, None]).sum(0) / np.maximum(g.sum(0), 1e-300) * (g.sum(0) > 1e-16) + 1e4 * (g.sum(0) <= 1e-16)
sp_c = lambda f: {"temperature": ScalarField(SampledField(jnp.asarray(T_x)[None])), "velocity": VectorField(VelocityRamp(0.0, 0.0, L20)),
                  "number_density": ScalarField(SampledField(jnp.asarray(f * n_x)[None])), "turbulence": ScalarField(Constant(0.0)),
                  "collisional": ScalarField(Constant(0.0))}
lya_si = LineBundle(["HI-LyA", "SiIII-1206"], ref_line_id="HI-LyA")
F_cl = Fields({"HI": sp_c(1.0), "SiIII": sp_c(1 / 5)})
a_pts = np.asarray(sbf.scale_factors[0, 0])
a_dag = 1 / (1 + z_obs)
P_box = float(phys.c_kms * np.log(a_pts.max() / a_pts.min()))                          # the path in velocity
dv_px = P_box / 2048
v_box = (np.arange(2048) + 0.5) * dv_px + float(phys.c_kms * np.log(a_dag / a_pts.max()))
tau_own = compute_optical_depth(sbf, v_box, lya_si, F_cl, a_dagger=a_dag)                            # (a) each line in its own frame
dV = float(lya_si.delta_v[1])
v_glob = np.arange(v_box[0] - dV - 60.0, v_box[-1] + 60.0, dv_px)                        # (b) one axis that holds both lines
tau_glob = compute_optical_depth(sbf, v_glob, lya_si, F_cl, a_dagger=a_dag, line_frame="reference")  # every line at v + Delta v_j
v_cl = [float(phys.c_kms * np.log(a_dag / np.interp(c0, xs, a_pts))) for c0, *_ in clumps]
fig, axs = plot_frames_axes(v_box, np.exp(-np.asarray(tau_own)[:, 0]), lya_si, v_ref=v_glob, F_ref=np.exp(-np.asarray(tau_glob)[:, 0]),
                            a_dagger=a_dag, a_obs=1.0, centres=v_cl)
add_panel_letters(axs)
fig.suptitle(r"The same four clumps in each line's own frame (a), on the global $\mathrm{Ly}\alpha$ axis (b) and in observed wavelength (c)",
             fontsize=21)
plt.show()
print(f"path: {P_box:.1f} km/s of Hubble flow; Delta v(Si III) = {dV:.1f} km/s; clump centres v_c = {np.round(v_cl, 1)} km/s "
      "(the same in both own frames)")
print(f"(b) Si III on the Lya axis = Si III evaluated at v + Delta v in its own frame: max |dtau| = "
      f"{np.abs(np.asarray(tau_glob)[1, 0] - np.asarray(compute_optical_depth(sbf, v_glob + dV, LineBundle(['SiIII-1206'], ref_line_id='SiIII-1206'), Fields({'SiIII': sp_c(1 / 5)}), a_dagger=a_dag))[0, 0]).max():.1e}")

*Reading the figure.*

- (a) In its own frame each line absorbs at the clump's $v_c$ (dotted): the same gas at the same velocity, in every
  line.
  - Each line's width is the clump's extent in Hubble flow plus its thermal width. The compact clumps make narrow lines,
    and the hot, extended one a broad, shallow line.
  - Si III is only slightly narrower than $\mathrm{Ly}\alpha$: its thermal width is 5.3 times smaller (Si is 28 times
    heavier), but the Hubble extent is the same for both lines.
- (b) On the $\mathrm{Ly}\alpha$ axis, Si III sits $\Delta v_j = 2270$ km/s to the blue of $\mathrm{Ly}\alpha$: the same
  clumps, the same profiles, moved by $-\Delta v_j$. The axis covers both lines, so nothing is cut.
- (c) On the observed-wavelength axis the two lines are $9.2\ \text{Å} \times (a_{\rm obs}/a_\dagger)$ apart.

## 6 · Periodicity: an option of the optical depth

**The fields are periodic; the spectrum is not.** A simulation box, like a toy GRF, is periodic in space: the gas past
its far face is the gas at its near face. The light that crosses the box once is absorbed by the Hubble flow from
$v_H = 0$ at one end to $v_H = P = c\ln(a_0/a_L)$ at the other, so the two ends of the box map to different velocities.
The gas at the near face absorbs near $v = 0$ and not near $P$, where the gas past the far face would absorb if the path
went on. The spectrum is therefore not naturally periodic: a periodic spectrum is a choice, and the default of the core
is the direct, non-periodic sum, for every source.

**The option.** Periodicity is one option of `compute_optical_depth`, `velocity_period`:

- `False` (the default): every piece absorbs where its line centre is; what falls off the grid is not on it.
- `True`: each skewer is periodic over its own velocity span, $P = c\,\lvert\ln(a_{\rm first}/a_{\rm last})\rvert$, set by
  the scale factors at its first and last points (`Pieces.velocity_span_kms`).
- A number: that period [km/s] for every skewer.

With a period $P$, every detuning $v_\dagger - v_c$ is replaced by its nearest image in $[-P/2, P/2)$. This one rule has
two distinct effects:

- **(i)** a piece whose line centre, Hubble flow plus peculiar velocity, falls outside the period absorbs at the other
  end, at $v_c - P$ (or $v_c + P$);
- **(ii)** a piece whose line centre is inside the period, but whose profile reaches past an end (the wings of a strong
  absorber near a face, even at rest), has those wings re-enter at the other end.

**Its limitation.** The exact periodic spectrum is the sum over all images, $\sum_n \tau_1(v_\dagger + nP)$; the nearest
image keeps one of them. The others matter only for damped wings: at half a period from a piece the two nearest images
are equally far, and one of them is dropped. For forest absorbers the missing $\tau$ is negligible; for a damped
absorber it is not (panel (e) and the numbers below).

The spectral grid is a separate choice. `SimulationSkewers.spectral_grid` covers $[0, P)$ of a box, but it implies no
wrap. The periodicity of the gas itself (the minimum images of the selection and the deposition, the periodic
interpolation of `GRFDensity`) is the topology of the gas, not an option of $\tau$.

**Step by step**, on five pieces in a 20 cMpc box at $z = 3$, one colour each: three forest absorbers, one near the
far face moving away from the observer (effect (i)), and a damped absorber at rest 0.4 cMpc from the far face
(effect (ii)).

In [ ]:
from skewerjax.utils import tutorial
from skewerjax.utils.plotting import plot_periodic_pieces
pp = tutorial.periodic_pieces_demo()        # five pieces in a 20 cMpc box at z = 3, observer at s = 0
fig, axs = plot_periodic_pieces(pp)
add_panel_letters(axs)
fig.suptitle("Where every piece absorbs: without wrap, and with velocity_period=True (nearest image modulo P)", fontsize=21)
plt.show()
v, P = pp["v"], pp["P"]
print(f"P = c ln(a(0)/a(L)) = {P:.1f} km/s")
for i in range(len(pp["s"])):
    print(f"piece {i}: s = {pp['s'][i]:5.1f} cMpc, log N = {pp['logN'][i]:4.1f}, u = {pp['u'][i]:+5.0f} km/s | v_H = {pp['v_H'][i]:7.1f}, "
          f"v_c = {pp['v_c'][i]:7.1f} km/s | peak: no wrap {v[np.argmax(pp['tau_nowrap'][i])]:7.1f}, wrapped {v[np.argmax(pp['tau_wrap'][i])]:7.1f} km/s")
j = pp["strong"]
miss = pp["tau_strong_exact"] - pp["tau_strong_nearest"]
print(f"damped piece: tau at v < 50 km/s: no wrap {pp['tau_nowrap'][j][v < 50].max():.2e}, wrapped {pp['tau_wrap'][j][v < 50].max():.2e}")
print(f"nearest image vs the exact periodic sum: max missing tau {miss.max():.3f} at v = {v[np.argmax(miss)]:.0f} km/s "
      f"(half a period from the piece), max |dF| = {np.abs(np.exp(-pp['tau_strong_exact']) - np.exp(-pp['tau_strong_nearest'])).max():.3f}")

**When to use which.**

- **No wrap** (the default) is the integral along the path as it is: right for any path that is not a single crossing
  of a periodic box (a light cone, a curved path, a segment of a longer sightline), and the cheapest. For a box it
  leaves out the absorption that falls beyond the ends: in 16 TNG50-4 skewers the mean flux is higher by about
  $7\times10^{-3}$ than with a periodic spectrum, and the P1D differs by up to 25 % at $k < 0.1$ s/km.
- **`velocity_period=True`** gives a spectrum periodic over the box, on a grid of one period. FFT-based estimators of a
  box need it (07). Its approximations: wrapped gas keeps its own scale factor (that of the face it belongs to), and
  only the nearest image of each piece is kept, which matters for damped wings in small boxes.

## 7 · Application: $\tau$ from simulation outputs

The same integrator on the gas of notebook 05. `SimulationSkewers.optical_depth(lines)` integrates every line in its
own frame over the velocity span of the box (1 km/s pixels by default), from the deposited pieces of every dual segment
(§1). It passes `velocity_period` through: the default is the direct, non-periodic sum (§6), and
`velocity_period=True` makes each skewer periodic over its span. `fields(species)` gives the same gas as a
`core.Fields` at the points.

In [ ]:
DATA = os.environ.get("SKEWERJAX_DATA", "/home/STORAGE")      # folder with TNG50-4/ (docs/installation.md)
from skewerjax.core import Cosmology
from skewerjax.simulations import ArepoHDF5Reader, SimulationSkewers, IntegrationOptions, HI
reader = ArepoHDF5Reader(f"{DATA}/TNG50-4", 25)
H = reader.header
cosmo = Cosmology(H0=100 * H.h * 1e6 * phys.Gyr_to_s / phys.Gpc_to_m, Omega_r_0=0.0, Omega_m_0=H.omega_m, Omega_k_0=0.0, Omega_lambda_0=H.omega_l)
L = H.box_cMpc
x = np.linspace(0.0, L, int(round(H.vmax_kms / 0.25)) + 1)
points = np.stack([np.stack([x, np.full_like(x, y), np.full_like(x, z)], -1) for y, z in [(10.0, 30.0), (25.0, 5.0), (40.0, 45.0)]])
bundle = SkewerBundle(spatial_points=jnp.asarray(points), a0=H.a, ICs_defined_at_index=0, ICs_at_emission_or_absorption="absorption", cosmologies=cosmo)
sk = SimulationSkewers.from_bundle(reader, bundle)
print(f"TNG50-4 snapshot 25: the three skewers of notebook 05, {len(sk.cells):,} cells selected in {sk.selection_s:.1f} s")

In [ ]:
lines = LineBundle(["HI-LyA", "SiIII-1206"], ref_line_id="HI-LyA")
velocities, tau = sk.optical_depth(lines)
flux = np.exp(-tau)
fields_sim = sk.fields(lines.species_keys)
sampled = fields_sim.sample(bundle.quadri_vectors[0], bundle.los_directions)
a_pts = np.asarray(bundle.quadri_vectors[0][..., 0])
v_pts = phys.c_kms * np.log(a_pts[:, :1] / a_pts)
fig, axs = plt.subplots(3, 3, figsize=(22, 12), sharex=True, sharey="row", layout="constrained")
for k in range(3):
    for j, l in enumerate(lines.lines):
        sty = MS["default"] if j == 0 else MS["matched"]
        axs[0, k].semilogy(v_pts[k], np.asarray(sampled[l.species]["number_density"])[k] / H.a ** 3, color=EC[l.species], **sty)
        axs[2, k].plot(velocities, flux[j, k], color=EC[l.species], **sty, label=l.id)
    axs[1, k].semilogy(v_pts[k], np.asarray(sampled["HI"]["temperature"])[k], color=EC["temperature"])
    axs[0, k].set_title(f"skewer {k}: y = {points[k, 0, 1]:.0f}, z = {points[k, 0, 2]:.0f} cMpc", fontsize=18); axs[2, k].set_xlabel("v [km/s]")
axs[0, 0].set_ylabel(r"$n$ [cm$^{-3}$]"); axs[1, 0].set_ylabel("T [K]"); axs[2, 0].set_ylabel("F"); axs[2, 0].legend()
add_panel_letters(axs, size=16); fig.suptitle("Three skewers through TNG50-4: the gas (top) and the flux it causes (bottom)", fontsize=22); plt.show()
print(f"<F>: Lya {flux[0].mean():.3f}, Si III {flux[1].mean():.4f}")